<a href="https://colab.research.google.com/github/William-HVH/Generativ-AI-with-LLMs/blob/main/Project03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instructions

This notebook contains the third project assignment. You work on the project in groups. You should already have formed a group. If not, do it on Brightspace under *My Courses > Groups*. To get started, do the following steps:

1) One student in the group makes a copy of this notebook.
2) Share it with your group using *Share* top-right. Here, add group members under people with access.
3) Moreover, under *Share > General access*, choose *Anyone with the link > Commenter*. Copy this link and paste it below.
4) Work with the notebook and solve the tasks.
5) Hand in the notebook by downloading it: File > Download > Download .ipynb. Next, on BS under *Project*, upload the file (I need that for the exam). Moreover, add the shared link of your Colab notebook as a link when you handin too.
6) After hand-in do peer grading (see BS under *Project*)

Sharing link: https://colab.research.google.com/drive/1PRaVnhYL1ELaXfEXdsvQSRiOIQXNqQsF?usp=sharing

**Deadlines**

* Hand-in solution 7/12/25
* Peer grading 13/12/25


# Notebook pre steps

In [ ]:
#@title Installations

# install missing packages
!pip install -q dfply

import numpy as np
import gdown
import gzip
import pandas as pd
import itertools
import joblib
import warnings
from dfply import *
from IPython.display import Markdown
from plotnine import *
from pprint import pprint
from scipy.stats import norm
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.4/612.4 kB 9.6 MB/s eta 0:00:00


# Problem description

Consider the [Hour-to-hour electricity market problem](https://colab.research.google.com/drive/1BFI6iRLTnrHgxgFayC89SKhuM3XcIp7s?usp=sharing) that we solved using a Markov decision process by discretising the state and action space.

Here, we are going to solve the problem using RL with approximation. How to solve the problem depends on which environment your are using. You may use one of the following environment for the problem:

1. The environment given in [Project 2](https://colab.research.google.com/drive/1HhQbV2ZvfMsEv-RkoGS6Sj0mjzC9QYF8?usp=sharing) which uses a discretization of both states and actions,
2. The environment given in the section "Environment with discrete actions" below.
3. The environment given in the section "Environment with continuous actions" below.

In all environments we represent state and actions using tuples $s_t = (s^l_t,s^c_t,s^w_t,s^p_t)$ and $a_t = (a^b_t, a^w_t, a^c_t)$. Moreover, we use settings:

# Environment with discrete actions

We here use an environment where

1. The battery only can be charged/discharged in units of $\delta$, e.g. $\delta = 1$ MWh. That is, $a^b_t \in \{-b^{\max}, \ldots, -\delta, 0, \delta, ..., b^{\max} \}$
2. Due to 1, the battery level is $s^l_t \in \{ 0, \delta, 2\delta, ..., b^{\max} \}$.
3. The same units are used for the wind energy production $a^w_t \in \{ 0, \delta, 2\delta, ..., p^{\max} \}$.
4. Due to 1-3, the commitment levels are $s^c_t \in \{ -b^\max, \ldots, -\delta, 0, \delta, \ldots, b^{\max} + p^{\max}\}$ since we can at most buy energy for an empty battery and we can at most sell max wind production plus a full battery. That is, $a^c_t$ have these commitment levels.

Note $\delta$ must satisfy that $b^{\max}/\delta$ and $p^{\max}/\delta$ are integer. Moreover, when calculating $E(x)$, its value must be rounded down so contained in the set defined for $a^w_t$.

To specify the discrete actions we only need to specify sets for the wind and commitment levels. Hence the battery levels will automatically be discrete too.

The environment is implemented below.


In [ ]:
class EnvEnergyDiscrete:
    """
    Environment for the hour-to-hour electricity market problem with discrete actions.
    """

    def __init__(self, settings, a_w_tuple, a_c_tuple, seed = 25328):
        """
        Initialize the environment.

        Args:
            settings (dict): A dictionary containing problem settings
            a_w_tuple (list): A list of possible wind usage
            a_c_tuple (list): A list of possible commitment levels


            seed (int): A seed for the random number generator
        """
        self.settings = settings

        ## price model
        # Google Drive direct download URL (using the 'uc?id=' format)
        url = 'https://drive.google.com/uc?id=1cGhte06iiWZnaRLPyj5D8ZzWX7gqPsIR'
        # Output filename for the downloaded file
        output_filename = 'prices_ar1.pkl.gz'
        # Download the file from Google Drive
        gdown.download(url, output_filename, quiet=True)
        # Load the model from the downloaded file
        with gzip.open(output_filename, "rb") as f:
            model_price = joblib.load(f)
        print(f"Price model loaded successfully from {output_filename}")
        self.model_price = model_price

        ## wind model
        # Google Drive direct download URL (using the 'uc?id=' format)
        url = 'https://drive.google.com/uc?id=1TJ1ACzev40QbeUlXBbDicYU3kEyiH1nB'
        # Output filename for the downloaded file
        output_filename = 'wind_log_ar1.pkl.gz'
        # Download the file from Google Drive
        gdown.download(url, output_filename, quiet=True)
        # Load the model from the downloaded file
        with gzip.open(output_filename, "rb") as f:
            model_wind = joblib.load(f)
        print(f"Wind model loaded successfully from {output_filename}")
        self.model_wind = model_wind

        self.rng = np.random.default_rng(seed)

        self.a_c_tuple = a_c_tuple
        self.a_w_tuple = a_w_tuple

    def reset_rng(self, seed):
        """
        Reset the random number generator.

        Args:
            seed (int): A seed for the random number generator
        """
        self.rng = np.random.default_rng(seed)

    def get_left(self, sorted_list, a):
        """
        Given a sorted list and a number 'a', finds the largest number 'b' in
        the list so b <= a

        Args:
            sorted_list (list): A list of numbers sorted in ascending order.
            a (float or int): The number to find the bounds for.

        Returns:
            float or int: The largest number 'b' in the list so b <= a.
        """
        # Iterate through the sorted list to find the bounds
        for i in range(len(sorted_list)-1, -1, -1):
            if sorted_list[i] <= a:
                return sorted_list[i]
        return None

    def get_actions(self, s: tuple) -> list:
        """
        Get all possible actions for a given state.
        """
        s_l, s_c, s_w, _ = s
        b_max = self.settings['b_max']
        # Determine a_b and a_w
        e_max = self.get_left(self.a_w_tuple, self.energy(s_w)) # max energy we can sell/produce (e.g. if energy(s_w) = 1.25 but a_w_tuple is [0,1,2,...] then can only sell 1)
        if s_c >= 0:
            if s_c >= e_max:
                a_b = min(s_l, s_c - e_max)
                a_w = e_max
            else: # s_c < e_max:
                a_b = -min(b_max - s_l, e_max - s_c)
                a_w = s_c - a_b
        else: # s_c < 0:
            a_b = -min(b_max - s_l, - s_c + e_max)
            a_w = max(0, - a_b + s_c)

        actions = [(float(a_b), float(a_w), float(a_c)) for a_c in self.a_c_tuple if a_c >= -(b_max - (s_l - a_b))]
        # if -(b_max - (s_l - a_b)) > -10:
        #     print(f"s_c: {s_c},  s_w: {s_w}, e_max: {e_max}")
        #     print(f"s_l: {s_l}, a_b: {a_b}, a_w: {a_w}, min a_c: {-(b_max - (s_l - a_b))}")
        #     print(min([a[2] for a in actions]))
        return actions

    def state_variables_to_str(self, s_l, s_c, s_w, s_p):
        """
        String representaion of a state.
        """
        return "{'s_l': " + str(s_l) + ", 's_c': " + str(s_c) + ", 's_w': " + str(s_w) + ", 's_p': " + str(s_p) + "}"

    def action_variables_to_str(self, a_b, a_w, a_c):
        """
        String representaion of an action.
        """
        return "{'a_b': " + str(a_b) + ", 'a_w': " + str(a_w) + ", 'a_c': " + str(a_c) + "}"

    def power(self, wind_speed):
        """
        The power output of a wind turbine given a wind speed.
        """
        p_max = self.settings['p_max']
        w_cut_in = self.settings['w_cut_in']
        w_rated = self.settings['w_rated']
        w_cut_out = self.settings['w_cut_out']

        if wind_speed < w_cut_in:
            return 0
        elif w_cut_in <= wind_speed <= w_rated:
            return p_max * ((wind_speed - w_cut_in) / (w_rated - w_cut_in)) ** 3
        elif w_rated < wind_speed <= w_cut_out:
            return p_max
        else:
            return 0

    def energy(self, w, time_period_length = 1):
        """
        The energy output of a wind turbine over one hour given a wind speed (MWh).
        """
        return self.power(w) * time_period_length

    def energy_inverse(self, energy_target, time_period_length = 1):
        """
        Find the wind speed that produces a given amount of energy, assuming wind is between w_cut_in and w_rated.

        Args:
            energy_target (float): The target energy output in MWh.
            time_period_length (float, optional): The time period length in hours. Defaults to 1.

        Returns:
            float: The wind speed that produces the given energy.
        """
        p_max = self.settings['p_max']
        w_cut_in = self.settings['w_cut_in']
        w_rated = self.settings['w_rated']

        if energy_target < 0 or energy_target > p_max:
            return None

        def find_wind_speed(w):
            return self.energy(w, time_period_length) - energy_target

        sol = root_scalar(find_wind_speed, bracket=[w_cut_in, w_rated], method='brentq')
        return sol.root if sol.converged else None


    def get_reward(self, s_p, s_c, a_b, a_w):
        """
        Calculate the reward for given state-action values.
        """
        c_plus = self.settings['c_plus']
        c_minus = self.settings['c_minus']
        e = a_b + a_w
        if s_p >= 0:
            return s_c * s_p - c_plus * abs(s_c - e)
        if s_p < 0:
            return s_c * s_p - c_minus * abs(s_c - e)
        raise ValueError("Reward can not be calculated.")

    def generate_price_next_hour(self, price):
        """
        Generates a random sample of the price for the next hour based on an AR(1) model.

        Args:
            price (float): The current price.

        Returns:
            float: A random sample of the next hour's price.
        """
        phi = self.model_price.params.iloc[1]
        intercept = self.model_price.params.iloc[0]
        sigma = self.model_price.resid.std()
        # The mean of the next price's distribution is the predicted value
        mean_next_price = intercept + phi * price
        # The standard deviation of the next price's distribution is the residual standard deviation
        std_dev_next_price = sigma
        # Generate a random sample from a normal distribution
        price_next_sample = self.rng.normal(loc=mean_next_price, scale=std_dev_next_price)
        return price_next_sample

    def generate_wind_next_hour(self, wind):
        """
        Generates a random sample of the wind speed for the next hour based on an AR(1) model.

        Args:
            wind (float): The current wind speed.

        Returns:
            float: A random sample of the next hour's wind speed.
        """
        # Add a small epsilon to prevent log(0) if wind is zero.
        # This ensures numerical stability for the log-transformed AR(1) model.
        min_positive_wind = 1e-6 # A very small positive number to avoid log(0)
        log_wind = np.log(max(wind, min_positive_wind))
        phi = self.model_wind.params.iloc[1]
        intercept = self.model_wind.params.iloc[0]
        sigma = self.model_wind.resid.std()
        # The mean of the next wind speed's distribution is the predicted value
        log_mean_next_wind = intercept + phi * log_wind
        # The standard deviation of the next wind speed's distribution is the residual standard deviation
        std_dev_next_wind = sigma
        # Generate a random sample from a normal distribution
        log_wind_next_sample = self.rng.normal(loc=log_mean_next_wind, scale=std_dev_next_wind)
        return np.exp(log_wind_next_sample)

    def get_step(self, s: tuple, a: tuple):
        """
        Compute one-step transition and reward from (s, a).

        Args:
            s: State label
            a: Action label

        Returns:
            tuple: (sN, r)
        """
        s_l, s_c, s_w, s_p = s
        a_b, a_w, a_c = a
        s_l_next = float(s_l - a_b)
        s_c_next = float(a_c)
        s_w_next = float(self.generate_wind_next_hour(s_w)) # update current values
        s_p_next = float(self.generate_price_next_hour(s_p))
        s_next = [s_l_next, s_c_next, s_w_next, s_p_next]
        reward = float(self.get_reward(s_p, s_c, a_b, a_w))
        return s_next, reward


Let us create an instance of the class:

In [ ]:
def generate_discrete_set(min_val, max_val, step):
    """
    Generates a set of discrete values from min_val to max_val with step size.

    Args:
        min_val (float): The minimum value in the set.
        max_val (float): The maximum value in the set.
        step (float): The step size between consecutive values.

    Returns:
        set: A set of discrete values {min_val,…,−𝛿,0,𝛿,...,max_val}.
    """
    # Ensure step is a divisor of min_val
    if not np.isclose(min_val % step, 0):
        raise ValueError("min_val argument must be divisible by step.")
    if not np.isclose(max_val % step, 0):
        raise ValueError("max_val argument must be divisible by step.")
    return np.arange(min_val, max_val + step, step)

# These settings could be changed (not in original code)
settings = {
    'b_max': 10,  # battery capacity (MWh)
    'p_max': 10,  # max power output wind (MW)
    'c_max': 10 + 10,  # max commitment (MWh) # b_max + p_max
    'w_cut_in': 3,  # cut in wind speed (m/s)
    'w_rated': 12,  # rated wind speed (m/s)
    'w_cut_out': 25,  # cut out wind speed (m/s)
    'c_plus': 50,  # EUR/MWh
    'c_minus': 50, # EUR/MWh
}

step_mwh = 2
b_max = settings['b_max']
p_max = settings['p_max']
c_max = settings['c_max']
a_w_tuple = generate_discrete_set(0, p_max, step_mwh)
a_c_tuple = generate_discrete_set(-b_max, c_max, step_mwh)

envD = EnvEnergyDiscrete(settings, a_w_tuple = a_w_tuple, a_c_tuple = a_c_tuple)

Price model loaded successfully from prices_ar1.pkl.gz
Wind model loaded successfully from wind_log_ar1.pkl.gz


## Q1

Consider the code below and explain the output.

In [ ]:
#@title Solution

import random

s = [4.0, 6.0, 12.4, 34.7] #  s = (s_l, s_c, s_w, s_p)
np.random.seed(0)
for _ in range(5):
    actions = envD.get_actions(s)  # a = (a_b, a_w, a_c)
    a = random.choice(actions)
    print(f"(s_l, s_c, s_w, s_p): {s}, (a_b, a_w, a_c): {a} ->", end = " ")
    s, r = envD.get_step(s, a)
    s = [round(v,1) for v in s]
    a = [round(v,0) for v in a]
    print(f"{s}, Rew: {r:1.1f}")


(s_l, s_c, s_w, s_p): [4.0, 6.0, 12.4, 34.7], (a_b, a_w, a_c): (-4.0, 10.0, 20.0) -> [8.0, 20.0, 7.9, 31.3], Rew: 208.2
(s_l, s_c, s_w, s_p): [8.0, 20.0, 7.9, 31.3], (a_b, a_w, a_c): (8.0, 0.0, 10.0) -> [0.0, 10.0, 48.0, 39.0], Rew: 26.0
(s_l, s_c, s_w, s_p): [0.0, 10.0, 48.0, 39.0], (a_b, a_w, a_c): (0.0, 0.0, 2.0) -> [0.0, 2.0, 16.0, 55.7], Rew: -110.0
(s_l, s_c, s_w, s_p): [0.0, 2.0, 16.0, 55.7], (a_b, a_w, a_c): (-8.0, 10.0, 8.0) -> [8.0, 8.0, 1.7, 56.9], Rew: 111.4
(s_l, s_c, s_w, s_p): [8.0, 8.0, 1.7, 56.9], (a_b, a_w, a_c): (8.0, 0.0, -2.0) -> [0.0, -2.0, 0.1, 69.6], Rew: 455.2


In [ ]:
display(Markdown("""
We start in the state where we have 4.0 MWh in battery, and the commitment levels
is 6 MW. The wind speed is 12.4 m/s and price 34.7. We then choose a random
action among the possible actions, and print. We obseve the reward and the
next state, which is printed. We observe a reward of 208, as this is the power
we sell $6 * 34.7 = 208.2$. This is repeated 5 times.

The next state we have a commitment of 20, as this was the commitment action
chosen in the previous period. It then chooses another commitment level at
random (10MW), and we observe the reward again. This is a low reward, as we have
a big imbalance, as we have a big imablance (commiting 8 when we have commited 20).
This continues for 5 iterations.

We do not generate wind energy in the third iteration, as it is above the
w_cut_out limit.
"""))


We start in the state where we have 4.0 MWh in battery, and the commitment levels
is 6 MW. The wind speed is 12.4 m/s and price 34.7. We then choose a random
action among the possible actions, and print. We obseve the reward and the 
next state, which is printed. We observe a reward of 208, as this is the power 
we sell $6 * 34.7 = 208.2$. This is repeated 5 times.

The next state we have a commitment of 20, as this was the commitment action
chosen in the previous period. It then chooses another commitment level at
random (10MW), and we observe the reward again. This is a low reward, as we have
a big imbalance, as we have a big imablance (commiting 8 when we have commited 20).
This continues for 5 iterations.

We do not generate wind energy in the third iteration, as it is above the
w_cut_out limit. 


# Environment with continuous actions

We here use an environment where both actions and states are continuous.

The environment is implemented below.


In [ ]:
class EnvEnergyCont:
    """
    Environment for the hour-to-hour electricity market problem with continuous actions.
    """

    def __init__(self, settings, seed = 25328):
        """
        Initialize the environment.

        Args:
            settings (dict): A dictionary containing problem settings
            seed (int): A seed for the random number generator
        """
        self.settings = settings

        ## price model
        # Google Drive direct download URL (using the 'uc?id=' format)
        url = 'https://drive.google.com/uc?id=1cGhte06iiWZnaRLPyj5D8ZzWX7gqPsIR'
        # Output filename for the downloaded file
        output_filename = 'prices_ar1.pkl.gz'
        # Download the file from Google Drive
        gdown.download(url, output_filename, quiet=True)
        # Load the model from the downloaded file
        with gzip.open(output_filename, "rb") as f:
            model_price = joblib.load(f)
        print(f"Price model loaded successfully from {output_filename}")
        self.model_price = model_price

        ## wind model
        # Google Drive direct download URL (using the 'uc?id=' format)
        url = 'https://drive.google.com/uc?id=1TJ1ACzev40QbeUlXBbDicYU3kEyiH1nB'
        # Output filename for the downloaded file
        output_filename = 'wind_log_ar1.pkl.gz'
        # Download the file from Google Drive
        gdown.download(url, output_filename, quiet=True)
        # Load the model from the downloaded file
        with gzip.open(output_filename, "rb") as f:
            model_wind = joblib.load(f)
        print(f"Wind model loaded successfully from {output_filename}")
        self.model_wind = model_wind

        self.rng = np.random.default_rng(seed)

    def reset_rng(self, seed):
        """
        Reset the random number generator.

        Args:
            seed (int): A seed for the random number generator
        """
        self.rng = np.random.default_rng(seed)

    def check_s(self, s):
        """
        Check if a state is valid.
        """
        s_l, s_c, s_w, _ = s
        assert s_l >= 0 and s_l <= self.settings['b_max']
        assert s_c >= -self.settings['b_max'] and s_c <= self.settings['c_max']
        assert s_w >= 0

    def check_a(self, a):
        """
        Check if an action is valid.
        """
        a_b, a_w, a_c = a
        assert a_b >= -self.settings['b_max'] and a_b <= self.settings['b_max']
        assert a_w >= 0
        assert a_c >= -self.settings['b_max'] and a_c <= self.settings['c_max']

    def get_action_info(self, s: tuple) -> tuple:
        """
        Get fixed actions for a given state and the minimum a_c value to choose.
        """
        self.check_s(s)
        s_l, s_c, s_w, _ = s
        b_max = self.settings['b_max']
        # Determine a_b and a_w
        e_max = self.energy(s_w) # max wind energy we can sell/produce
        if s_c >= 0:
            if s_c >= e_max:
                a_b = min(s_l, s_c - e_max)
                a_w = e_max
            else: # s_c < e_max:
                a_b = -min(b_max - s_l, e_max - s_c)
                a_w = s_c - a_b
        else: # s_c < 0:
            a_b = -min(b_max - s_l, - s_c + e_max)
            a_w = max(0, - a_b + s_c)
        # print(f"s_c: {s_c},  s_w: {s_w}, e_max: {e_max}")
        # print(f"s_l: {s_l}, a_b: {a_b}, a_w: {a_w}, min a_c: {-(b_max - (s_l - a_b))}")
        a_c_min = -(b_max - (s_l - a_b))  # no need to buy power that cannot be stored
        return float(a_b), float(a_w), float(a_c_min)

    def state_variables_to_str(self, s_l, s_c, s_w, s_p):
        """
        String representaion of a state.
        """
        return "{'s_l': " + str(s_l) + ", 's_c': " + str(s_c) + ", 's_w': " + str(s_w) + ", 's_p': " + str(s_p) + "}"

    def action_variables_to_str(self, a_b, a_w, a_c):
        """
        String representaion of an action.
        """
        return "{'a_b': " + str(a_b) + ", 'a_w': " + str(a_w) + ", 'a_c': " + str(a_c) + "}"

    def power(self, wind_speed):
        """
        The power output of a wind turbine given a wind speed.
        """
        p_max = self.settings['p_max']
        w_cut_in = self.settings['w_cut_in']
        w_rated = self.settings['w_rated']
        w_cut_out = self.settings['w_cut_out']

        if wind_speed < w_cut_in:
            return 0
        elif w_cut_in <= wind_speed <= w_rated:
            return p_max * ((wind_speed - w_cut_in) / (w_rated - w_cut_in)) ** 3
        elif w_rated < wind_speed <= w_cut_out:
            return p_max
        else:
            return 0

    def energy(self, w, time_period_length = 1):
        """
        The energy output of a wind turbine over one hour given a wind speed (MWh).
        """
        return self.power(w) * time_period_length

    def get_reward(self, s_p, s_c, a_b, a_w):
        """
        Calculate the reward for given state-action values.
        """
        c_plus = self.settings['c_plus']
        c_minus = self.settings['c_minus']
        e = a_b + a_w
        if s_p >= 0:
            return s_c * s_p - c_plus * abs(s_c - e)
        if s_p < 0:
            return s_c * s_p - c_minus * abs(s_c - e)
        raise ValueError("Reward can not be calculated.")

    def generate_price_next_hour(self, price):
        """
        Generates a random sample of the price for the next hour based on an AR(1) model.

        Args:
            price (float): The current price.

        Returns:
            float: A random sample of the next hour's price.
        """
        phi = self.model_price.params.iloc[1]
        intercept = self.model_price.params.iloc[0]
        sigma = self.model_price.resid.std()
        # The mean of the next price's distribution is the predicted value
        mean_next_price = intercept + phi * price
        # The standard deviation of the next price's distribution is the residual standard deviation
        std_dev_next_price = sigma
        # Generate a random sample from a normal distribution
        price_next_sample = self.rng.normal(loc=mean_next_price, scale=std_dev_next_price)
        return price_next_sample

    def generate_wind_next_hour(self, wind):
        """
        Generates a random sample of the wind speed for the next hour based on an AR(1) model.

        Args:
            wind (float): The current wind speed.

        Returns:
            float: A random sample of the next hour's wind speed.
        """
        # Add a small epsilon to prevent log(0) if wind is zero.
        # This ensures numerical stability for the log-transformed AR(1) model.
        min_positive_wind = 1e-6 # A very small positive number to avoid log(0)
        log_wind = np.log(max(wind, min_positive_wind))
        phi = self.model_wind.params.iloc[1]
        intercept = self.model_wind.params.iloc[0]
        sigma = self.model_wind.resid.std()
        # The mean of the next wind speed's distribution is the predicted value
        log_mean_next_wind = intercept + phi * log_wind
        # The standard deviation of the next wind speed's distribution is the residual standard deviation
        std_dev_next_wind = sigma
        # Generate a random sample from a normal distribution
        log_wind_next_sample = self.rng.normal(loc=log_mean_next_wind, scale=std_dev_next_wind)
        return np.exp(log_wind_next_sample)

    def get_step(self, s: tuple, a: tuple):
        """
        Compute one-step transition and reward from (s, a).

        Args:
            s: State label
            a: Action label

        Returns:
            tuple: (sN, r)
        """
        self.check_s(s)
        self.check_a(a)
        s_l, s_c, s_w, s_p = s
        a_b, a_w, a_c = a
        s_l_next = float(s_l - a_b)
        s_c_next = float(a_c)
        s_w_next = float(self.generate_wind_next_hour(s_w)) # update current values
        s_p_next = float(self.generate_price_next_hour(s_p))
        s_next = [s_l_next, s_c_next, s_w_next, s_p_next]
        self.check_s(s_next)
        reward = float(self.get_reward(s_p, s_c, a_b, a_w))
        return s_next, reward


Let us create an instance of the class:

In [ ]:
envC = EnvEnergyCont(settings)

Price model loaded successfully from prices_ar1.pkl.gz
Wind model loaded successfully from wind_log_ar1.pkl.gz


## Q2

Consider the code below and explain the output.

In [ ]:
#@title Solution

import random

s = [4.6, 12.7, 12.4, 34.7] #  s = (s_l, s_c, s_w, s_p)
for _ in range(5):
    a_b, a_w, a_c_min = envC.get_action_info(s)  # a = (a_b, a_w, a_c)
    a_c = random.uniform(a_c_min, c_max)
    a = (a_b, a_w, a_c)
    a = [round(v,1) for v in a]
    print(f"(s_l, s_c, s_w, s_p): {s}, (a_b, a_w, a_c): {a} ->", end = " ")
    s, r = envC.get_step(s, a)
    s = [round(v,1) for v in s]
    print(f"{s}, Rew: {r:1.1f}")

(s_l, s_c, s_w, s_p): [4.6, 12.7, 12.4, 34.7], (a_b, a_w, a_c): [2.7, 10.0, 0.2] -> [1.9, 0.2, 12.4, 25.5], Rew: 440.7
(s_l, s_c, s_w, s_p): [1.9, 0.2, 12.4, 25.5], (a_b, a_w, a_c): [-8.1, 8.3, 13.7] -> [10.0, 13.7, 14.3, 55.8], Rew: 5.1
(s_l, s_c, s_w, s_p): [10.0, 13.7, 14.3, 55.8], (a_b, a_w, a_c): [3.7, 10.0, 8.2] -> [6.3, 8.2, 1.6, 46.7], Rew: 764.5
(s_l, s_c, s_w, s_p): [6.3, 8.2, 1.6, 46.7], (a_b, a_w, a_c): [6.3, 0.0, -1.6] -> [0.0, -1.6, 16.1, 25.5], Rew: 287.9
(s_l, s_c, s_w, s_p): [0.0, -1.6, 16.1, 25.5], (a_b, a_w, a_c): [-10.0, 8.4, 13.5] -> [10.0, 13.5, 12.4, 57.4], Rew: -40.8


In [ ]:
display(Markdown("""
The code is very similar, but in this case we have a continuous action space,
meaning that we can choose decimal numbers for how much power we sell and buy.
We therefore choose a random decimal between the minimum possible commitment
and the maximum commitment possible. The decimal is then rounded to .1.

We start in the state with a battery level of 4.6 and a commitment of 12.7.
We then choose to sell 2.7 from the battery and 10 from the wind, which leaves
us without an imbalance, and we have a reward of $440.7. The next period
we only choose to commit 0.2, which is a low number, and we therefore only
have a small reward here. As we have excess wind energy, we charge the battery
by 8.1MW.

This is repeated 5 times, and we end in a state where we have a battery
level of 10MW, and a commitment level of 13.5MW.
"""))


The code is very similar, but in this case we have a continuous action space,
meaning that we can choose decimal numbers for how much power we sell and buy.
We therefore choose a random decimal between the minimum possible commitment
and the maximum commitment possible. The decimal is then rounded to .1.

We start in the state with a battery level of 4.6 and a commitment of 12.7.
We then choose to sell 2.7 from the battery and 10 from the wind, which leaves
us without an imbalance, and we have a reward of $440.7. The next period
we only choose to commit 0.2, which is a low number, and we therefore only
have a small reward here. As we have excess wind energy, we charge the battery
by 8.1MW.

This is repeated 5 times, and we end in a state where we have a battery
level of 10MW, and a commitment level of 13.5MW. 


# Q3 - Solving the problem

Our objective is to maximise the average reward. Pick one of the environments above and solve the problem. Note in theory by having continuous actions, you don't limit your action space and you should be able to get a better objective value.

For solving the problem, you have different options:

1. Implement classes for state-value and policy approximation and apply an actor-critic algorithm, similar to what we have done in the course.
2. Convert the environment into a Gym-style RL environment so it can be used with an RL library. Then, utilise the RL library to solve the problem. Note that in most RL libraries, the objective is discounted reward, which we have argued is not the most suitable choice for continuing problems. However, if you use a discount rate close to 1 (such as 0.9999), you can mimic the average reward objective.

State and clearly connect your choices to (also holds if you use Deep RL):

* The average reward objective.
* Theory regarding state-value approximations.
* Theory about policy approximations.
* Theory used which has not been presented in the course.

ChatGPT (or similar) can be useful for initial guidance, but remember to critically evaluate and understand the reasoning behind your choices. If you require GPU acceleration in Colab, then change the runtime type to GPU-T4 (the dropdown beside RAM in the upper right corner).




In [ ]:
import numpy as np
from typing import Sequence, List, Optional


class SoftmaxPolicy:
    """Softmax policy with a separate parameter vector for each action.

    Each action a has its own weight vector θ_a. The preference function is:

        h(s, a) = θ_a^T x(s)

    where the state feature vector is:

        x(s) = [1.0, q / max_inv, t / max_t].

    This formulation supports state-dependent action sets, including a
    terminal scrap action.
    """

    def __init__(
        self,
        env: object,
        step_size: float,
        decay_fct: float = 1.0,
        rng_seed: Optional[int] = None,
    ) -> None:
        """Initializes the softmax policy.

        Args:
            env: Environment providing attributes `max_inv`, `max_t`,
                `prices`, `scrap_price`, and a method `get_actions(s)`.
            step_size: Initial learning rate alpha.
            decay_fct: Multiplicative decay applied to alpha after each
                update. Defaults to 1.0 (no decay).
            rng_seed: Optional random seed.
        """
        self.env = env
        self.max_inv = env.max_inv
        self.max_t = env.max_t

        # Global action space = price actions + scrap action
        self.prices: List[float] = list(env.prices)
        self.scrap_price: float = env.scrap_price
        self.all_actions: List[float] = self.prices + [self.scrap_price]

        # Explicit action → index mapping
        self.act2idx = {a: i for i, a in enumerate(self.all_actions)}
        self.num_actions = len(self.all_actions)

        # Learning rates
        self.initial_alpha = float(step_size)
        self.alpha = float(step_size)
        self.decay_fct = float(decay_fct)

        # RNG
        self.rng_seed = rng_seed
        self.rng = np.random.default_rng(rng_seed)

        # State feature count: [1, q/max_inv, t/max_t]
        self.num_features = 3

        # Parameter matrix: one θ-vector per action
        self.theta = np.zeros(
            (self.num_actions, self.num_features),
            dtype=float,
        )

    # ------------------------------------------------------------------

    def reset(self) -> None:
        """Resets policy parameters and random number generator."""
        self.theta[:] = 0.0
        self.alpha = self.initial_alpha
        self.rng = np.random.default_rng(self.rng_seed)

    @staticmethod
    def softmax(x: Sequence[float]) -> np.ndarray:
        """Computes a numerically stable softmax.

        Args:
            x: Input score vector.

        Returns:
            Softmax probabilities over the input.
        """
        x_arr = np.asarray(x, dtype=float)
        exp_x = np.exp(x_arr - np.max(x_arr))
        return exp_x / np.sum(exp_x)

    # ------------------------------------------------------------------

    def get_state_features(self, s: Sequence[int]) -> np.ndarray:
        """Constructs the state feature vector.

        Args:
            s: State [q, t].

        Returns:
            Feature vector x(s).
        """
        q, t = s
        return np.array(
            [1.0, q / self.max_inv, t / self.max_t],
            dtype=float,
        )

    # ------------------------------------------------------------------

    def get_action_preferences(self, s: Sequence[int]) -> np.ndarray:
        """Computes the preference values h(s, a) for available actions.

        Args:
            s: State [q, t].

        Returns:
            A vector of preferences for actions available in s.
        """
        x_s = self.get_state_features(s)
        actions = self.env.get_actions(s)

        # Convert actions to their global parameter indices
        idxs = [self.act2idx[a] for a in actions]

        return np.array([self.theta[i] @ x_s for i in idxs])

    # ------------------------------------------------------------------

    def get_probabilities(self, s: Sequence[int]) -> np.ndarray:
        """Computes the policy distribution π(. | s).

        Args:
            s: State [q, t].

        Returns:
            Action probabilities for actions available in s.
        """
        prefs = self.get_action_preferences(s)
        return self.softmax(prefs)

    # ------------------------------------------------------------------

    def get_action(self, s: Sequence[int]) -> float:
        """Samples an action from the policy.

        Args:
            s: State [q, t].

        Returns:
            Sampled action from π(. | s).
        """
        actions = self.env.get_actions(s)
        probs = self.get_probabilities(s)
        return float(self.rng.choice(actions, p=probs))

    def get_best_action(self, s: Sequence[int]) -> float:
        """Returns the action with the highest preference h(s, a).

        Args:
            s: State [q, t].

        Returns:
            The best action according to current θ.
        """
        actions = self.env.get_actions(s)
        prefs = self.get_action_preferences(s)
        return float(actions[int(np.argmax(prefs))])

    # ------------------------------------------------------------------

    def get_log_gradient(self, s: Sequence[int], a: float) -> np.ndarray:
        """Computes ∇θ log π(a | s).

        The gradient has the same shape as θ. For softmax:

            ∇θ_a log π(a | s)     = (1 − π(a | s)) · x(s)
            ∇θ_b log π(a | s)     = −π(b | s) · x(s),  b ≠ a

        Args:
            s: State [q, t].
            a: Chosen action.

        Returns:
            Gradient matrix matching shape of θ.
        """
        x_s = self.get_state_features(s)
        actions = self.env.get_actions(s)
        probs = self.get_probabilities(s)

        grad = np.zeros_like(self.theta)

        # Identify chosen action coordinate
        local_idx = actions.index(a)
        global_idxs = [self.act2idx[x] for x in actions]

        # Fill gradient rows
        for li, (gi, p) in enumerate(zip(global_idxs, probs)):
            if li == local_idx:
                grad[gi] = (1.0 - p) * x_s
            else:
                grad[gi] = -p * x_s

        return grad

    # ------------------------------------------------------------------

    def train(self, s: Sequence[int], a: float, mult: float) -> None:
        """Applies a REINFORCE parameter update.

        Update rule:
            θ ← θ + α * mult * ∇θ log π(a | s)

        Args:
            s: State at time t.
            a: Action taken at time t.
            mult: Scalar multiplier, typically (γ^t) G_t.
        """
        grad = self.get_log_gradient(s, a)
        self.theta += self.alpha * mult * grad
        self.alpha *= self.decay_fct


In [ ]:
#@title Actor critic (continuing tasks)
from typing import Any
def actor_critic_continuing(
    pi_hat,
    v_hat,
    steps: int,
    alpha_r: float,
    r_bar_ini: float = 0.0,
    callback = None,
    callback_every: int = 0,
    trace: Any = None,
) -> None:
    """
    Differential (average-reward) one-step actor–critic for continuing tasks.

    This version assumes a continuing task with no terminal states, and uses
    the average-reward formulation:

        δ_t = R_{t+1} - r̄_t + v_hat(S_{t+1}) - v_hat(S_t)

    Critic (TD(0)):
        w ← w + α_v δ_t ∇_w v_hat(S_t)

    Actor:
        θ ← θ + α_θ δ_t ∇_θ log π(A_t | S_t)

    Average reward estimate:
        r̄_{t+1} = r̄_t + α_r δ_t

    Args:
        pi_hat: Policy (actor) with get_action(s) and train(s, a, mult).
        v_hat: Value function (critic) with value(s) and train(s, target).
               train(s, target) should move v_hat(s) toward `target`.
        steps: Total number of time steps to run.
        alpha_r: Step size for updating the average reward r̄.
        callback: Optional logging callback.
        callback_every: Call callback every this many steps (if > 0).
        trace: Passed through to callback.
    """

    # Initialize
    s = pi_hat.env.reset()
    r_bar = r_bar_ini

    total_reward = 0.0
    total_reward_every = 0.0

    for t in tqdm(range(1, steps + 1)):
        # Sample action from current policy
        a = pi_hat.get_action(s)

        # Environment step (continuing task: no 'done' flag)
        s_next, r = pi_hat.env.get_step(s, a)

        # Critic: TD error for differential value
        v_s = v_hat.eval(s)
        v_s_next = v_hat.eval(s_next)
        delta_t = r - r_bar + v_s_next - v_s

        # Actor: update with δ_t
        mult = delta_t
        pi_hat.train(s, a, mult)

        # Critic: semi-gradient TD(0) toward v_s + δ_t
        td_target = v_s + delta_t
        v_hat.train(s, td_target)

        # Average reward update
        r_bar += alpha_r * delta_t

        # Bookkeeping
        total_reward += r
        total_reward_every += r

        # Optional logging callback (in terms of steps, not episodes)
        if (
            callback is not None
            and callback_every > 0
            and t % callback_every == 0
        ):
            info = {
                "step": t,
                "avg_reward_estimate": r_bar,
                "avg_reward_overall": total_reward / t,
                "avg_reward_every": total_reward_every / callback_every,
            }
            total_reward_every = 0.0
            callback(pi_hat, info, trace)

        # Move to next state
        s = s_next
    print("Final average reward estimate:", round(r_bar,2))
